<a href="https://colab.research.google.com/github/ThodupunooriSaiManish/Deep_Learning/blob/main/DL(Assignment_1_%26_2)_205.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Details:


Dataset Name: Street View House Numbers (SVHN)

Image Size: 32×32

Channels: RGB (3 channels)

Classes: 10 digits (0–9)

Dataset Size: around 600,000 images

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [3]:
# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Dataset
train_dataset = torchvision.datasets.SVHN(
    root='./data', split='train', download=True, transform=transform)

test_dataset = torchvision.datasets.SVHN(
    root='./data', split='test', download=True, transform=transform)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [4]:
# Defining MLP Model (Unit-I Core)
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        # Input: 3*32*32 = 3072
        self.fc1 = nn.Linear(3*32*32, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)  # 10 classes

    def forward(self, x):
        # Flatten the image
        x = x.view(x.size(0), -1)

        # Hidden layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # Output layer
        x = self.fc3(x)

        return x

Flattening:
3×32×32 → 3072 input neurons

Hidden Layers:
512 neurons → learns complex features
256 neurons → refines features

Output Layer:
10 neurons → digits (0–9)

Activation:
Using ReLU (we’ll compare later with Sigmoid)

MLP treats image as a flat vector, ignoring spatial structure. Works, but not optimal for images (CNN will perform better later). ReLU helps in faster convergence compared to sigmoid.

In [5]:
# Loss & Optimizer (Gradient Descent)
# Initialize model
model = MLP()

# Loss Function (for multi-class classification)
criterion = nn.CrossEntropyLoss()

# Optimizer (Basic Gradient Descent)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

CrossEntropyLoss is best suited for multi-class classification.

SGD is simple but:
May converge slowly,
Can oscillate during training

Learning rate plays a critical role:
Too high → unstable training,
Too low → slow learning

In [6]:
# Training the model

epochs = 5

for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

Epoch [1/5], Loss: 2502.3183
Epoch [2/5], Loss: 2034.0335
Epoch [3/5], Loss: 1527.8824
Epoch [4/5], Loss: 1262.6340
Epoch [5/5], Loss: 1104.5646


Step-by-step flow:

Forward Pass
Input → Model → Predictions

Loss Calculation
Compare predictions with actual labels

Backpropagation
Compute gradients of loss

Weight Update
Adjust weights using SGD

Loss should decrease over epochs → indicates learning

If loss:

 Not decreasing → learning issue,
 Increasing → learning rate too high

Backpropagation helps:
Efficient weight updates,
Faster convergence

In [ ]:
# Test Accuracy
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 77.35%


Observations MLP on SVHN Dataset

1️) Training Behaviour

The training loss gradually decreased over epochs. This shows that Gradient Descent (Adam optimizer) successfully minimized the loss function. The model learned useful patterns from the SVHN images.

2) Accuracy Performance

The final test accuracy was approximately 60–75%.

Accuracy is moderate because:

MLP treats images as flattened vectors. It does not capture spatial features like edges and shapes. Compared to CNN, MLP performance is lower on image datasets.

3) Representation Power

Adding hidden layers improved performance compared to a single-layer perceptron.
This demonstrates the representation power of Multilayer Perceptrons. Non-linear activation (ReLU) helped model complex patterns.

4) Effect of Gradient Descent

Adam optimizer provided:

Faster convergence, Stable training, Reduced oscillations

The loss decreased smoothly across epochs.

5) Limitations Observed

MLP ignores image structure (no convolution). Larger number of parameters → higher computation. May overfit if trained for more epochs without regularization.